# Train SimpleGPT on a Machine Learning Textbook

**Goal:** Train your TinyGPT on a mini machine-learning textbook written as **question → answer pairs**, then "ask" it questions and watch it generate the answers.

**What you will learn:** A language model does not truly *understand* questions — it only predicts the next word. Because every question in the training text is always followed by its answer, prompting the model with a question makes it complete the pattern with the answer. This is the core idea behind how large language models appear to answer questions.

**Files needed:** `ml_textbook.txt` and `My_transformer.py` in the same folder as this notebook.

In [48]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from My_transformer import Trans

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using:", device)

Using: cuda


## Step 1 — Load the textbook

Instead of typing sentences by hand, we read the whole mini textbook from a file. Each line is one question and its answer, ending with `<END>`.

In [49]:
with open("ml_textbook.txt", "r", encoding="utf-8") as f:
    text = f.read().replace("\n", " ").strip()

print(text[:300], "...")

what is machine learning ? machine learning means computers learn patterns from data <END> what is a model ? a model is a program that learns patterns from data <END> what is training ? training means the model learns from examples in the data <END> what is a dataset ? a dataset is a collection of e ...


## Step 2 — Build the vocabulary

Same as before: split into words, build `word2idx` and `idx2word`. This corpus has ~407 tokens and ~119 unique words — bigger than the 10-sentence corpus, so the model gets a slightly harder job.

In [50]:
words = sorted(set(text.split()))   # sorted -> same ordering every run
vocab_size = len(words)
print("Vocab size:", vocab_size)

word2idx = {w: i for i, w in enumerate(words)}
idx2word = {i: w for w, i in word2idx.items()}

data = torch.tensor([word2idx[w] for w in text.split()], dtype=torch.long)
print("Total tokens:", len(data))

Vocab size: 142
Total tokens: 491


## Step 3 — Hyperparameters

Because the textbook is bigger than the 10-sentence corpus, we scale up a little:

| Parameter | Old | New | Why |
|---|---|---|---|
| `block_size` | 6 | 12 | answers are longer — the model must see the whole question while writing the answer |
| `embedding_dim` | 32 | 64 | 119 words need richer vectors than 42 words |
| `epochs` | 1500 | 3000 | more data takes longer to memorize |

In [51]:
block_size = 12
embedding_dim = 64
n_heads = 2
n_layers = 2
lr = 1e-3
epochs = 3000

def get_batch(batch_size=16):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

## Step 4 — The model (same TinyGPT as before)

In [52]:
class SimpeGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, embedding_dim)
        self.position_embedding = nn.Embedding(block_size, embedding_dim)
        self.blocks = nn.Sequential(*[Trans(embedding_dim, block_size, n_heads) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(embedding_dim)
        self.head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            next_idx = torch.multinomial(probs, 1)
            idx = torch.cat((idx, next_idx), dim=1)
        return idx

## Step 5 — Train

Loss should drop from ~4.8 toward ~0.1. If your final loss is still above 0.5, run this cell again (it continues training) or increase `epochs`.

In [53]:
model = SimpeGPT().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

for step in range(epochs):
    xb, yb = get_batch()
    xb, yb = xb.to(device), yb.to(device)
    logits, loss = model(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 300 == 0:
        print(f"Step {step}, loss={loss.item():.4f}")
print(f"Final loss: {loss.item():.4f}")

Step 0, loss=5.0771
Step 300, loss=0.6483
Step 600, loss=0.3177
Step 900, loss=0.2211
Step 1200, loss=0.3020
Step 1500, loss=0.2590
Step 1800, loss=0.2180
Step 2100, loss=0.2606
Step 2400, loss=0.1448
Step 2700, loss=0.2261
Final loss: 0.2378


## Step 6 — Ask the model a question!

We write a small helper: it encodes your question, lets the model continue the text, and cuts the output at `<END>`.

**Important:** every word in your question must exist in the textbook vocabulary, and the question should match the textbook's style (`what is X ?`). Words are case-sensitive and `?` is a separate token.

In [54]:
def ask(question, max_new_tokens=20):
    q_tokens = question.split()
    unknown = [w for w in q_tokens if w not in word2idx]
    if unknown:
        print("These words are not in the vocabulary:", unknown)
        return
    context = torch.tensor([[word2idx[w] for w in q_tokens]], dtype=torch.long).to(device)
    out = model.generate(context, max_new_tokens=max_new_tokens)
    result = [idx2word[int(i)] for i in out[0]]
    answer = result[len(q_tokens):]              # only the generated part
    if "<END>" in answer:
        answer = answer[:answer.index("<END>")]  # stop at <END>
    print("Q:", question)
    print("A:", " ".join(answer))

ask("what is overfitting ?")

Q: what is overfitting ?
A: overfitting means the model memorizes training data and fails on new data


In [55]:
# Try more questions:
ask("what is a transformer ?")
ask("what is gradient descent ?")
ask("what is a loss function ?")

Q: what is a transformer ?
A: a transformer is a neural network that uses attention to learn from sequences
Q: what is gradient descent ?
A: gradient descent updates weights step by step to reduce the loss
Q: what is a loss function ?
A: a loss function measures how wrong the weights and biases and an activation function to produce an output


In [56]:
# Ask the model all 25 textbook questions:
ask("what is machine learning ?")
ask("what is a model ?")
ask("what is training ?")
ask("what is a dataset ?")
ask("what is a feature ?")
ask("what is a label ?")
ask("what is supervised learning ?")
ask("what is unsupervised learning ?")
ask("what is a prediction ?")
ask("what is overfitting ?")
ask("what is underfitting ?")
ask("what is a loss function ?")
ask("what is an epoch ?")
ask("what is a learning rate ?")
ask("what is a neural network ?")
ask("what is a weight ?")
ask("what is gradient descent ?")
ask("what is a test set ?")
ask("what is classification ?")
ask("what is regression ?")
ask("what is an embedding ?")
ask("what is a transformer ?")
ask("what is attention ?")
ask("what is a token ?")
ask("what is generalization ?")
ask("what is reinforcement learning ?")
ask("what is deep learning ?")
ask("what is a neuron ?")
ask("what is an activation function ?")
ask("what is preprocessing ?")

Q: what is machine learning ?
A: machine learning means computers learn patterns from data
Q: what is a model ?
A: a model is a program that learns patterns from data
Q: what is training ?
A: training means the model learns from examples in the data
Q: what is a dataset ?
A: a dataset is a collection of examples used for training
Q: what is a feature ?
A: a feature is an input value that describes an example
Q: what is a label ?
A: a label is the correct answer for an example
Q: what is supervised learning ?
A: supervised learning means training with features and labels together
Q: what is unsupervised learning ?
A: unsupervised learning means finding patterns in data without labels
Q: what is a prediction ?
A: a prediction is the output the model gives for new data
Q: what is overfitting ?
A: overfitting means the model memorizes training data and fails on new data
Q: what is underfitting ?
A: underfitting means the model is too simple to learn the patterns
Q: what is a loss function 

## Step 7 — Tasks

1. **Ask all 25 questions** from the textbook. How many does the model answer correctly? Why does a low loss mean correct answers here?
2. **Ask a question that is NOT in the textbook**, using only vocabulary words — for example `what is a dataset model ?`. What happens? Why is the output nonsense?
3. **Break it:** ask `What is overfitting ?` with a capital W. Explain the error you get.
4. **Experiment:** set `epochs = 300` (undertrained), retrain, and ask again. What changes?
5. **Extend the textbook:** add 5 new Q&A lines to `ml_textbook.txt` in the same style, retrain, and test your new questions.
6. **Discussion:** this model *memorized* answers — it cannot answer anything new. What would it need to answer questions it has never seen?

In [57]:
ask("what is a dataset model ?")

Q: what is a dataset model ?
A: a dataset is a collection of examples used for training


In [58]:
ask("What is overfitting ?")

These words are not in the vocabulary: ['What']
